**calc95pct.ipynb**
- Calculates mean daily temperature (tas) climatology for 1979-2000 using AUS-11 (BARRA-R2).

**Reads:** "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest/" \
**Writes:** "ID_HW_BARRA/data/preprocess/t95_baseline.nc" \
**Compute:** xxlarge (28CPU, 126GB) \
**Environment:** analysis3

In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client, wait
from datetime import datetime

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 28,Total memory: 126.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36017,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46731,Total threads: 4
Dashboard: /proxy/38107/status,Memory: 18.00 GiB
Nanny: tcp://127.0.0.1:39693,


2025-12-05 11:43:12,099 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 16d193512fd2cd457c21244fe5866dde initialized by task ('rechunk-merge-rechunk-split-rechunk-transfer-a9775ba85df9b94d40fd9ad1247f52f4', 0, 0, 0, 21, 0, 0) executed on worker tcp://127.0.0.1:45741
2025-12-05 11:43:12,306 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 56b21fe1c3f050420cb76e5d0e828702 initialized by task ('rechunk-merge-rechunk-split-rechunk-transfer-a9775ba85df9b94d40fd9ad1247f52f4', 0, 0, 1, 21, 0, 1) executed on worker tcp://127.0.0.1:45741
2025-12-05 11:43:12,354 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f690f485bb3e17854d8bee150242e750 initialized by task ('rechunk-merge-rechunk-split-rechunk-transfer-a9775ba85df9b94d40fd9ad1247f52f4', 0, 1, 0, 21, 1, 0) executed on worker tcp://127.0.0.1:45741
2025-12-05 11:43:14,154 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 1f09fc522ae5ac09a9cd84dc7a3b329c initialized by task ('rechunk-merge-rechunk-tr

In [4]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/tas/latest/"
write_path = f'{workingDir}/data/preprocess/'

In [5]:
sdate, edate ='19790101', '20001231'

In [6]:
fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
fnames = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fnames])

tas_ds = xr.open_mfdataset(fpaths, concat_dim ='time', combine='nested', 
                           parallel=True, data_vars='minimal',coords='minimal', 
                           drop_variables = "time_bnds", chunks="auto")
datestr = f"s{sdate}_e{edate}"

t95 = tas_ds.reduce(np.nanpercentile, q=95, dim="time")
t95 = t95.rename(name_dict={'tas':'PRCTILE95'})

/jobfs/156016945.gadi-pbs/ipykernel_457362/971153490.py:5: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  tas_ds = xr.open_mfdataset(fpaths, concat_dim ='time', combine='nested',
/jobfs/156016945.gadi-pbs/ipykernel_457362/971153490.py:5: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  tas_ds = xr.open_mfdataset(fpaths, concat_dim ='time'

In [7]:
# Minimal fix 1: clear stale encodings
try:
    t95.encoding.clear()
    for v in t95.data_vars:
        t95[v].encoding.clear()
except Exception:
    pass

encoding = {"PRCTILE95":{"zlib": True, "complevel": 4, "shuffle": True}}

In [8]:
t95 = t95.persist()
wait(t95)           # ensure reduction finished on workers
t95 = t95.compute() # materialize to memory to avoid 'tas' backend refs
tas_ds.close()

t95.to_netcdf(f'{write_path}t95_baseline.nc',
              encoding=encoding,
              engine='netcdf4')